In [1]:
import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("ERROR: GPU not available!")

print("\nGPU Details:")
subprocess.run(["nvidia-smi"])

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU name: Tesla T4
GPU count: 1

GPU Details:


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/AI651_A2/Question2_colab.zip"
extract_path = "/content/AI651"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully!")
print("Files:", os.listdir(extract_path))

ZIP extracted successfully!
Files: ['Question2']


In [4]:
for root, dirs, files in os.walk("/content/AI651"):
    print(root)
    for file in files:
        print("  ", file)

/content/AI651
/content/AI651/Question2
   run_all.sh
   train.py
   verify_runs.py
   COLAB_CELLS.md
   select_d_cfg.py
   checks.py
   select_d.py
   mm.py
   choose_d.py
   extra_analysis.py
   analysis.py


In [5]:
import os

project = "/content/AI651/Question2"
drive_base = "/content/drive/MyDrive/AI651_A2"

os.chdir(project)

for folder in ["runs", "results", "gifs"]:
    drive_folder = os.path.join(drive_base, folder)
    local_link = os.path.join(project, folder)

    os.makedirs(drive_folder, exist_ok=True)

    if os.path.islink(local_link):
        os.unlink(local_link)
    elif os.path.exists(local_link):
        raise RuntimeError(
            f"{local_link} already exists. Check it before continuing."
        )

    os.symlink(drive_folder, local_link)

print("Working directory:", os.getcwd())

for folder in ["runs", "results", "gifs"]:
    print(folder, "->", os.path.realpath(folder))

Working directory: /content/AI651/Question2
runs -> /content/drive/MyDrive/AI651_A2/runs
results -> /content/drive/MyDrive/AI651_A2/results
gifs -> /content/drive/MyDrive/AI651_A2/gifs


In [6]:
for folder in ["runs", "results", "gifs"]:
    assert os.path.islink(folder), f"{folder} is not a symlink"
    assert os.path.isdir(folder), f"{folder} is not accessible"

print("SUCCESS: All folders connected to Google Drive!")

SUCCESS: All folders connected to Google Drive!


In [7]:
%cd /content/AI651/Question2

!pip install -q opencv-python-headless pillow matplotlib torchvision

/content


In [8]:
import torch
import mm

print("CUDA available:", torch.cuda.is_available())
print("Project device:", mm.DEVICE)

assert torch.cuda.is_available(), "GPU not available!"

videos = mm.load_videos("data")

print("Dataset shape:", videos.shape)
print("Dataset dtype:", videos.dtype)
print("Dataset loaded successfully!")

CUDA available: True
Project device: cuda


100%|██████████| 819M/819M [00:20<00:00, 39.9MB/s]


Dataset shape: torch.Size([10000, 20, 1, 64, 64])
Dataset dtype: torch.uint8
Dataset loaded successfully!


In [9]:
!python checks.py

(a) max|M(M(x)) - x|                                           expected 0                                      yours 0, torch.equal=True
(b) max|R^4(x) - x|                                            expected 0                                      yours 0, torch.equal=True
(c) max|M R M(x) - R^-1(x)|                                    expected 0                                      yours 0, torch.equal=True
(d) bright pixel of M(p) (frame, row, column)                  expected (7, 10, 43)                            yours (7, 10, 43)
(e) bright pixel of R(p) (frame, row, column)                  expected (7, 43, 10)                            yours (7, 43, 10)
(f) 1x input / target frames                                   expected 1-5 / 6-10                             yours 1-5 / 6-10
(g) 2x input / target frames                                   expected 1, 3, 5, 7, 9 / 11, 13, 15, 17, 19     yours 1, 3, 5, 7, 9 / 11, 13, 15, 17, 19
(h) 3x input / target frames                       

In [10]:
import mm

model = mm.build_model("A").to(mm.DEVICE)

print("Model device:", next(model.parameters()).device)
print("Total parameters:", mm.n_params(model))

assert mm.n_params(model) == 480784

print("Model A verification passed!")

Model device: cuda:0
Total parameters: 480784
Model A verification passed!


In [11]:
%cd /content/AI651/Question2

!bash run_all.sh check

/content
cv2 4.14.0 | torchvision 0.26.0+cu130 | numpy 2.1.3
torch 2.11.0+cu130 | cuda True | device cuda
Tesla T4
dataset (10000, 20, 1, 64, 64) torch.uint8
(a) max|M(M(x)) - x|                                           expected 0                                      yours 0, torch.equal=True
(b) max|R^4(x) - x|                                            expected 0                                      yours 0, torch.equal=True
(c) max|M R M(x) - R^-1(x)|                                    expected 0                                      yours 0, torch.equal=True
(d) bright pixel of M(p) (frame, row, column)                  expected (7, 10, 43)                            yours (7, 10, 43)
(e) bright pixel of R(p) (frame, row, column)                  expected (7, 43, 10)                            yours (7, 43, 10)
(f) 1x input / target frames                                   expected 1-5 / 6-10                             yours 1-5 / 6-10
(g) 2x input / target frames                 

In [12]:
!python train.py --help

usage: train.py [-h] --model {A,B,C,D,DB} --seed SEED [--epochs EPOCHS]
                [--threads THREADS] [--out OUT] [--skip_done]
                [--smoke_steps SMOKE_STEPS]

options:
  -h, --help            show this help message and exit
  --model {A,B,C,D,DB}
  --seed SEED
  --epochs EPOCHS
  --threads THREADS     CPU threads (0 = PyTorch default)
  --out OUT
  --skip_done           exit immediately if this run is already complete in
                        --out
  --smoke_steps SMOKE_STEPS
                        TEST ONLY: stop each epoch after this many batches


In [13]:
%cd /content/AI651/Question2

!bash run_all.sh A

/content
A_s0 already complete -> skipped
A_s1 already complete -> skipped
{
 "blank": {
  "mse_h": [
   0.04261038452386856,
   0.0425695925951004,
   0.04257640987634659,
   0.04253190755844116,
   0.04252973943948746
  ],
  "E5_1x": 0.042563606053590775,
  "E2_1x": 0.04258999973535538,
  "params": 0
 },
 "copy": {
  "mse_h": [
   0.04916374385356903,
   0.06041966751217842,
   0.06688860058784485,
   0.07077877968549728,
   0.0731603354215622
  ],
  "E5_1x": 0.06408222019672394,
  "E2_1x": 0.05479172617197037,
  "params": 0
 },
 "A_0": {
  "mse_h": [
   0.017206203192472458,
   0.020696328952908516,
   0.022987887263298035,
   0.024717221036553383,
   0.026488803327083588
  ],
  "E5_1x": 0.022419288754463196,
  "E2_1x": 0.01895126886665821,
  "params": 480784,
  "train_s_per_epoch": 30.029693093499986,
  "epochs": 30
 },
 "A_1": {
  "mse_h": [
   0.016792241483926773,
   0.02016555704176426,
   0.02246086113154888,
   0.024183088913559914,
   0.02577057108283043
  ],
  "E5_1x": 0.02

In [14]:
%cd /content/AI651/Question2

!bash run_all.sh selectD

/content
{
 "candidate": "p8_k3",
 "params": 594496,
 "steps": 1500,
 "seconds": 42.180188096999984,
 "E5_1x": 0.03859748691320419,
 "E2_1x": 0.03862134367227554,
 "E5_2x": 0.038595762103796005,
 "E2_2x": 0.038594432175159454,
 "E5_3x": 0.017371835187077522,
 "E2_3x": 0.038588330149650574,
 "score": 0.11578024923801422
}
{
 "candidate": "p4_k5_h32",
 "params": 359184,
 "steps": 1500,
 "seconds": 97.89575732199995,
 "E5_1x": 0.039013221859931946,
 "E2_1x": 0.03903951868414879,
 "E5_2x": 0.03900928050279617,
 "E2_2x": 0.03900991752743721,
 "E5_3x": 0.017170893028378487,
 "E2_3x": 0.039003174751996994,
 "score": 0.11702631413936615
}
model D design (validation only): p8_k3 {'p8_k3': 0.11578024923801422, 'p4_k5_h32': 0.11702631413936615}


In [15]:
import json

with open("/content/AI651/Question2/results/d_choice.json") as f:
    choice = json.load(f)

print(json.dumps(choice, indent=2))

{
  "patch": 8,
  "hidden": 64,
  "kernels": [
    3,
    3
  ],
  "dilations": [
    1,
    1
  ],
  "chosen": "p8_k3",
  "scores": {
    "p8_k3": 0.11578024923801422,
    "p4_k5_h32": 0.11702631413936615
  },
  "screening_steps": 1500
}


In [16]:
%cd /content/AI651/Question2

!bash run_all.sh BCD

/content
B_s0 already complete -> skipped
B_s1 already complete -> skipped
C_s0 params 480784 device Tesla T4 | torch 2.11.0+cu130
C_s0 epoch 1/30 loss 0.04661 time 30.9s
C_s0 epoch 2/30 loss 0.03995 time 31.3s
C_s0 epoch 3/30 loss 0.03937 time 29.9s
C_s0 epoch 4/30 loss 0.03859 time 30.5s
C_s0 epoch 5/30 loss 0.03766 time 30.4s
C_s0 epoch 6/30 loss 0.03662 time 30.4s
C_s0 epoch 7/30 loss 0.03586 time 30.4s
C_s0 epoch 8/30 loss 0.03499 time 30.4s
C_s0 epoch 9/30 loss 0.03430 time 30.4s
C_s0 epoch 10/30 loss 0.03392 time 30.4s
C_s0 epoch 11/30 loss 0.03344 time 30.4s
C_s0 epoch 12/30 loss 0.03312 time 30.4s
C_s0 epoch 13/30 loss 0.03289 time 30.5s
C_s0 epoch 14/30 loss 0.03277 time 30.5s
C_s0 epoch 15/30 loss 0.03245 time 30.4s
C_s0 epoch 16/30 loss 0.03240 time 30.4s
C_s0 epoch 17/30 loss 0.03221 time 30.4s
C_s0 epoch 18/30 loss 0.03199 time 30.5s
C_s0 epoch 19/30 loss 0.03192 time 30.4s
C_s0 epoch 20/30 loss 0.03167 time 30.4s
C_s0 epoch 21/30 loss 0.03146 time 30.4s
C_s0 epoch 22/30 

In [17]:
%cd /content/AI651/Question2

!bash run_all.sh eval

/content
OK      A seed 0: epochs 30/30 params 480784 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 30.0s final loss 0.02230
OK      A seed 1: epochs 30/30 params 480784 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 30.1s final loss 0.02193
OK      B seed 0: epochs 30/30 params 480784 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 30.3s final loss 0.02355
OK      B seed 1: epochs 30/30 params 480784 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 30.5s final loss 0.02217
OK      C seed 0: epochs 30/30 params 480784 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 30.5s final loss 0.03015
OK      C seed 1: epochs 30/30 params 480784 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 30.4s final loss 0.03139
OK      D seed 0: epochs 30/30 params 594496 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 9.3s final loss 0.03012
OK      D seed 1: epochs 30/30 params 594496 device cuda (Tesla T4 | torch 2.11.0+cu130) mean epoch 9.3s final loss 0